# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainabaon/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zainabaon/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
import numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape)

(30000, 45)


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_df = df.copy()

# Engineered features
feature_df["log_impressions"] = np.log1p(feature_df["impressions_90d"])
feature_df["is_stale"] = (feature_df["days_since_last_update"] >= 180).astype(int)
feature_df["is_visible"] = (feature_df["impressions_90d"] >= 500).astype(int)

# Categorical handling: one-hot encode position_tier and content_type
feature_df = pd.get_dummies(feature_df, columns=["position_tier", "content_type"], prefix=["pos", "type"])

# Fill missing values (word_count has NaNs seen earlier)
feature_df["word_count"] = feature_df["word_count"].fillna(feature_df["word_count"].median())
feature_df["ctr"] = feature_df["ctr"].fillna(0)

final_features = ["content_age_days", "days_since_last_update", "impressions_90d",
                   "log_impressions", "avg_position", "ctr", "word_count",
                   "is_stale", "is_visible"] + [c for c in feature_df.columns if c.startswith("pos_") or c.startswith("type_")]

print("Final feature count:", len(final_features))
feature_df[final_features].head()

Final feature count: 17


,content_age_days,days_since_last_update,impressions_90d,log_impressions,avg_position,ctr,word_count,is_stale,is_visible,pos_deep,pos_page_1,pos_page_3_5,pos_striking,pos_top_3,type_comparison article,type_feedly article,type_keyword article
0,187,20,3803,8.243808,10.6,0.76,3221.0,0,1,False,False,False,True,False,False,False,True
1,445,25,15320,9.636980,20.3,0.05,2481.0,0,1,False,False,True,False,False,False,False,True
2,141,20,12581,9.440023,36.5,0.09,3515.0,0,1,False,False,True,False,False,False,False,True
3,463,22,11751,9.371779,6.2,0.49,2877.0,0,1,False,True,False,False,False,False,False,True
4,263,14,19140,9.859588,44.0,0.13,2803.0,0,1,False,False,True,False,False,False,False,True


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

- content_age_days — how old the page is in days. Missing: none observed. Available-when: known at any point after publish, always before decision time.
- days_since_last_update — staleness signal. Missing: none observed. Available-when: known at decision time.
- impressions_90d / log_impressions — trailing 90-day visibility. Missing: none observed; log version reduces skew from a few very high-traffic pages. Available-when: a trailing window measurement, known at decision time.
- avg_position — average search ranking position. Missing: none observed. Available-when: known at decision time.
- ctr — click-through rate. Missing: filled with 0 (no clicks recorded). Available-when: known at decision time, derived from same-window clicks/impressions.
- word_count — page length. Missing: filled with median, since ~15% of rows had NaN. Available-when: known at any point after content is published.
- is_stale / is_visible — engineered binary flags from days_since_last_update and impressions_90d. Available-when: same as their source columns, decision-time known.
- pos_* / type_* (one-hot) — position tier and content type category dummies. Available-when: known at decision time, these are descriptive/categorical facts about the page.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Attack 1: check if trend_pct (label-derived) accidentally exists in final_features
print("trend_pct in final_features?", "trend_pct" in final_features)

# Attack 2: check correlation of each feature with the label to spot suspiciously perfect ones
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X = feature_df[final_features].select_dtypes(include=[np.number, bool]).fillna(0)
y = feature_df["is_declining_label"]

model = LogisticRegression(max_iter=1000).fit(X, y)
auc = roc_auc_score(y, model.predict_proba(X)[:,1])
print("AUC with current feature set:", round(auc, 4))
print("(A suspiciously high AUC like 0.99+ would signal leakage; this is a healthy, moderate number)")

# Attack 3: deliberately add a leaky column and show the AUC jump (same demo as w03_data_contract)
X_leak = X.copy()
X_leak["leaky"] = y * 100
model_leak = LogisticRegression(max_iter=1000).fit(X_leak, y)
auc_leak = roc_auc_score(y, model_leak.predict_proba(X_leak)[:,1])
print("AUC WITH a deliberately leaky column added:", round(auc_leak, 4), "<- confirms the leakage test works")

trend_pct in final_features? False


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


AUC with current feature set: 0.6688
(A suspiciously high AUC like 0.99+ would signal leakage; this is a healthy, moderate number)
AUC WITH a deliberately leaky column added: 1.0 <- confirms the leakage test works


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- trend_pct — directly determines trend_direction (which the label is built from); including it would let the model read the answer back directly.
- trend_direction — this IS the label source; using it as a feature would be using the answer as an input.
- Any FlyRank product decision fields (health_score, priority_score, action_type, refresh flags) — not shipped in this dataset, and even if reconstructed, would only teach a model to copy an existing rule instead of finding real signal.
- content_id / client_id — join keys/identifiers only, carry no real predictive signal themselves and risk the model memorizing specific IDs rather than learning generalizable patterns.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.